<a href="https://colab.research.google.com/github/mohankvcs25-spec/BIS_LAB_PROGRMAS/blob/main/ACO_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

import random

# ==============================================
# 1. HOSPITAL DETAILS
# ==============================================

random.seed(42)

beds = 3
doctors = 3
nurses = 3
ventilators = 1

patients = [
    {"id": 1, "type": "Emergency", "priority": 10, "ventilator": True},
    {"id": 2, "type": "Urgent", "priority": 5, "ventilator": False},
    {"id": 3, "type": "Normal", "priority": 1, "ventilator": False},
    {"id": 4, "type": "Emergency", "priority": 10, "ventilator": False},
    {"id": 5, "type": "Urgent", "priority": 5, "ventilator": False}
]

N = len(patients)

# ==============================================
# 2. ACO PARAMETERS
# ==============================================

NUM_ANTS = 20
ITERATIONS = 100

ALPHA = 1.0
BETA = 2.0

EVAPORATION = 0.5
Q = 100

# Pheromone matrix
pheromone = [[1.0] * N for _ in range(N + 1)]


# ==============================================
# 3. CONSTRUCT SOLUTION
# ==============================================

def construct_solution():

    visited = set()
    solution = []

    current = N  # Starting position

    for _ in range(N):

        choices = []
        weights = []

        for j in range(N):

            if j not in visited:

                priority = patients[j]["priority"]

                heuristic = priority / 10.0

                value = (
                    pheromone[current][j] ** ALPHA
                    * heuristic ** BETA
                )

                choices.append(j)
                weights.append(value)

        selected = random.choices(
            choices,
            weights=weights,
            k=1
        )[0]

        solution.append(selected)
        visited.add(selected)

        current = selected

    return solution


# ==============================================
# 4. EVALUATE SOLUTION
# ==============================================

def evaluate_solution(solution):

    available_beds = beds
    available_doctors = doctors
    available_nurses = nurses
    available_ventilators = ventilators

    total_cost = 0
    total_waiting_time = 0

    emergency_allocated = 0
    urgent_allocated = 0
    normal_allocated = 0

    allocation = []

    for position, index in enumerate(solution):

        patient = patients[index]

        pid = patient["id"]
        category = patient["type"]
        priority = patient["priority"]
        needs_ventilator = patient["ventilator"]

        # Check resource availability

        can_allocate = (
            available_beds > 0
            and available_doctors > 0
            and available_nurses > 0
            and (
                not needs_ventilator
                or available_ventilators > 0
            )
        )

        if can_allocate:

            # Allocate resources

            available_beds -= 1
            available_doctors -= 1
            available_nurses -= 1

            if needs_ventilator:
                available_ventilators -= 1

            # OBJECTIVE 1: RESOURCE COST

            cost = 10 + 8 + 5

            if needs_ventilator:
                cost += 20

            total_cost += cost

            # OBJECTIVE 2: WAITING TIME

            waiting_time = position * 10

            total_waiting_time += waiting_time

            # OBJECTIVE 3: EMERGENCY PRIORITY

            if category == "Emergency":
                emergency_allocated += 1

            elif category == "Urgent":
                urgent_allocated += 1

            else:
                normal_allocated += 1

            status = "Allocated"

        else:

            # Patient waits for the next allocation period

            waiting_time = 60

            total_waiting_time += waiting_time

            status = "Waiting"

        allocation.append({
            "id": pid,
            "type": category,
            "priority": priority,
            "status": status,
            "waiting_time": waiting_time
        })

    # Count emergency patients

    total_emergency = sum(
        1 for p in patients
        if p["type"] == "Emergency"
    )

    total_urgent = sum(
        1 for p in patients
        if p["type"] == "Urgent"
    )

    total_normal = sum(
        1 for p in patients
        if p["type"] == "Normal"
    )

    emergency_waiting = (
        total_emergency - emergency_allocated
    )

    unallocated = (
        N - emergency_allocated
        - urgent_allocated - normal_allocated
    )

    # Combined fitness function

    fitness = (
        total_cost
        + total_waiting_time
        + emergency_waiting * 1000
        + (unallocated - emergency_waiting) * 200
    )

    return {
        "fitness": fitness,
        "cost": total_cost,
        "waiting": total_waiting_time,
        "emergency_allocated": emergency_allocated,
        "urgent_allocated": urgent_allocated,
        "normal_allocated": normal_allocated,
        "emergency_waiting": emergency_waiting,
        "allocation": allocation
    }


# ==============================================
# 5. PHEROMONE UPDATE
# ==============================================

def update_pheromone(solution, fitness):

    global pheromone

    # Evaporation

    for i in range(N + 1):

        for j in range(N):

            pheromone[i][j] *= (1 - EVAPORATION)

    # Deposit pheromone on the best solution

    deposit = Q / (1 + fitness)

    previous = N

    for patient_index in solution:

        pheromone[previous][patient_index] += deposit

        previous = patient_index


# ==============================================
# 6. MAIN ACO ALGORITHM
# ==============================================

best_solution = None
best_fitness = float("inf")

for iteration in range(ITERATIONS):

    iteration_best_solution = None
    iteration_best_fitness = float("inf")

    for ant in range(NUM_ANTS):

        solution = construct_solution()

        result = evaluate_solution(solution)

        fitness = result["fitness"]

        if fitness < iteration_best_fitness:

            iteration_best_fitness = fitness
            iteration_best_solution = solution[:]

        if fitness < best_fitness:

            best_fitness = fitness
            best_solution = solution[:]

    update_pheromone(
        iteration_best_solution,
        iteration_best_fitness
    )


# ==============================================
# 7. FINAL RESULT
# ==============================================

result = evaluate_solution(best_solution)

total_emergency = sum(
    p["type"] == "Emergency" for p in patients
)

total_urgent = sum(
    p["type"] == "Urgent" for p in patients
)

total_normal = sum(
    p["type"] == "Normal" for p in patients
)

total_allocated = sum(
    p["status"] == "Allocated"
    for p in result["allocation"]
)

total_waiting = N - total_allocated


# ==============================================
# 8. DISPLAY HOSPITAL INFORMATION
# ==============================================

print("\n" + "=" * 55)
print("       HOSPITAL RESOURCE ALLOCATION")
print("       ANT COLONY OPTIMIZATION")
print("=" * 55)

print("\n1. HOSPITAL INFORMATION")

print("Total Patients       :", N)
print("Emergency Patients   :", total_emergency)
print("Urgent Patients      :", total_urgent)
print("Normal Patients      :", total_normal)


# ==============================================
# 9. AVAILABLE RESOURCES
# ==============================================

print("\n2. AVAILABLE RESOURCES")

print("Hospital Beds        :", beds)
print("Doctors              :", doctors)
print("Nurses               :", nurses)
print("Ventilators          :", ventilators)


# ==============================================
# 10. PATIENT PRIORITY
# ==============================================

print("\n3. PATIENT PRIORITY DETAILS")

for p in patients:

    print(
        "Patient", p["id"],
        "|", p["type"],
        "| Priority:", p["priority"]
    )

print("\nPriority Scale")

print("Emergency = 10 (Highest)")
print("Urgent    = 5  (Medium)")
print("Normal    = 1  (Low)")


# ==============================================
# 11. THREE OBJECTIVES
# ==============================================

print("\n" + "=" * 55)
print("             THREE OBJECTIVES")
print("=" * 55)

# Objective 1

print("\nOBJECTIVE 1: MINIMIZE RESOURCE ALLOCATION COST")

print("Total Allocation Cost :", result["cost"])


# Objective 2

print("\nOBJECTIVE 2: REDUCE PATIENT WAITING TIME")

print("Total Waiting Time    :", result["waiting"], "minutes")


# Objective 3

print("\nOBJECTIVE 3: EMERGENCY PATIENT PRIORITY")

print("Total Emergency Patients :",
      total_emergency)

print("Emergency Allocated      :",
      result["emergency_allocated"])

print("Emergency Waiting        :",
      result["emergency_waiting"])


# ==============================================
# 12. FINAL PATIENT ALLOCATION
# ==============================================

print("\n" + "=" * 55)
print("              FINAL ALLOCATION")
print("=" * 55)

for p in result["allocation"]:

    print(
        "Patient", p["id"],
        "|", p["type"],
        "| Priority:", p["priority"],
        "|", p["status"],
        "| Waiting:", p["waiting_time"], "minutes"
    )


# ==============================================
# 13. FINAL SUMMARY
# ==============================================

print("\n" + "=" * 55)
print("              FINAL SUMMARY")
print("=" * 55)

print("Total Patients       :", N)
print("Patients Allocated   :", total_allocated)
print("Patients Waiting     :", total_waiting)

print("Best Fitness Value   :", result["fitness"])

print("\nACO Parameters")

print("Number of Ants       :", NUM_ANTS)
print("Iterations           :", ITERATIONS)
print("Alpha                :", ALPHA)
print("Beta                 :", BETA)
print("Evaporation Rate     :", EVAPORATION)

print("\nOptimization Completed Successfully!")

print("=" * 55)



       HOSPITAL RESOURCE ALLOCATION
       ANT COLONY OPTIMIZATION

1. HOSPITAL INFORMATION
Total Patients       : 5
Emergency Patients   : 2
Urgent Patients      : 2
Normal Patients      : 1

2. AVAILABLE RESOURCES
Hospital Beds        : 3
Doctors              : 3
Nurses               : 3
Ventilators          : 1

3. PATIENT PRIORITY DETAILS
Patient 1 | Emergency | Priority: 10
Patient 2 | Urgent | Priority: 5
Patient 3 | Normal | Priority: 1
Patient 4 | Emergency | Priority: 10
Patient 5 | Urgent | Priority: 5

Priority Scale
Emergency = 10 (Highest)
Urgent    = 5  (Medium)
Normal    = 1  (Low)

             THREE OBJECTIVES

OBJECTIVE 1: MINIMIZE RESOURCE ALLOCATION COST
Total Allocation Cost : 89

OBJECTIVE 2: REDUCE PATIENT WAITING TIME
Total Waiting Time    : 150 minutes

OBJECTIVE 3: EMERGENCY PATIENT PRIORITY
Total Emergency Patients : 2
Emergency Allocated      : 2
Emergency Waiting        : 0

              FINAL ALLOCATION
Patient 4 | Emergency | Priority: 10 | Allocated | 